# 06 SHAP Explainability

This notebook applies SHAP analysis to interpret the best-performing XGBoost model.

The goal is to identify which airline service features most strongly influence passenger satisfaction predictions.

## Imports and Preprocessing

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
except ImportError:
    print("Please install xgboost: pip install xgboost")

RANDOM_STATE = 42

import shap
import matplotlib.pyplot as plt



# Update these paths if your CSV files are stored elsewhere.
train_path = "../data/train.csv"
test_path = "../data/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Remove unnamed index columns if present
train_df = train_df.loc[:, ~train_df.columns.str.contains("^Unnamed")]
test_df = test_df.loc[:, ~test_df.columns.str.contains("^Unnamed")]

# Impute missing arrival delay values using the training median
arrival_delay_median = train_df["Arrival Delay in Minutes"].median()
train_df["Arrival Delay in Minutes"] = train_df["Arrival Delay in Minutes"].fillna(arrival_delay_median)
test_df["Arrival Delay in Minutes"] = test_df["Arrival Delay in Minutes"].fillna(arrival_delay_median)

# Encode categorical variables
nominal_cols = ["Gender", "Type of Travel"]
ordinal_cols = ["Customer Type", "Class"]

ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
oe = OrdinalEncoder(
    categories=[
        ["disloyal Customer", "Loyal Customer"],
        ["Eco", "Eco Plus", "Business"]
    ]
)
le = LabelEncoder()

train_nominal = pd.DataFrame(
    ohe.fit_transform(train_df[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=train_df.index
)

test_nominal = pd.DataFrame(
    ohe.transform(test_df[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=test_df.index
)

train_ordinal = pd.DataFrame(
    oe.fit_transform(train_df[ordinal_cols]),
    columns=ordinal_cols,
    index=train_df.index
)

test_ordinal = pd.DataFrame(
    oe.transform(test_df[ordinal_cols]),
    columns=ordinal_cols,
    index=test_df.index
)

train_encoded = train_df.drop(columns=nominal_cols + ordinal_cols).join(train_nominal).join(train_ordinal)
test_encoded = test_df.drop(columns=nominal_cols + ordinal_cols).join(test_nominal).join(test_ordinal)

train_encoded["satisfaction"] = le.fit_transform(train_encoded["satisfaction"])
test_encoded["satisfaction"] = le.transform(test_encoded["satisfaction"])

# Separate features and target
X_train = train_encoded.drop(columns=["satisfaction", "id"])
y_train = train_encoded["satisfaction"]

X_test = test_encoded.drop(columns=["satisfaction", "id"])
y_test = test_encoded["satisfaction"]

# Scale features for models that benefit from standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data shape:", X_train.shape)
print("Test data shape:", X_test.shape)
print("Target classes:", list(le.classes_))

## Train Final XGBoost Model

In [ ]:
best_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=RANDOM_STATE
)

best_xgb.fit(X_train_scaled, y_train)

print("Final XGBoost model trained.")

## SHAP Summary Plot

In [ ]:
# Convert scaled arrays back to DataFrames for readable SHAP plots
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# Use a sample for faster explanation
X_test_sample = X_test_scaled_df.sample(100, random_state=RANDOM_STATE)

explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_test_sample)

shap.summary_plot(shap_values, X_test_sample, show=False)
plt.tight_layout()
plt.savefig("../figures/shap_summary_plot.png", dpi=300, bbox_inches="tight")
plt.show()

## Top Feature Importance

In [ ]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)

feature_importance = pd.DataFrame({
    "Feature": X_test_sample.columns,
    "Mean_ABS_SHAP": mean_abs_shap
}).sort_values(by="Mean_ABS_SHAP", ascending=False)

top_features = feature_importance.head(10)
display(top_features)

top_features.to_csv("../results/top_shap_features.csv", index=False)

ax = top_features.sort_values("Mean_ABS_SHAP").plot(
    x="Feature",
    y="Mean_ABS_SHAP",
    kind="barh",
    figsize=(8, 6),
    legend=False
)
ax.set_title("Top SHAP Features for Passenger Satisfaction")
ax.set_xlabel("Mean Absolute SHAP Value")
plt.tight_layout()
plt.savefig("../figures/feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

## Interpretation

The strongest drivers of passenger satisfaction were:

1. Inflight WiFi Service
2. Customer Type
3. Online Boarding
4. Leg Room Service
5. Arrival Delay in Minutes

These results suggest that airlines can improve satisfaction by prioritizing digital service quality, loyalty experience, boarding processes, seat comfort, and delay management.